# Corrigé — Jour 2 Matin : RAG et MCP

## TP 3 Partie A — Mini-RAG amélioré

In [ ]:
# ========================================================================
# RAG ameliore : chunking par mots, chargement depuis dossier .md, citation
# ========================================================================

# pip install sentence-transformers numpy
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

# --- 1. Chunking par fenetre glissante (par mots)
def chunk_text(texte, taille=200, overlap=40):
    """Decoupe un texte en morceaux de 'taille' mots, avec recouvrement."""
    mots = texte.split()
    morceaux = []
    debut = 0
    while debut < len(mots):
        # On prend une fenetre de 'taille' mots a partir de 'debut'
        fin = debut + taille
        morceaux.append(" ".join(mots[debut:fin]))
        # On avance de (taille - overlap) mots pour la prochaine fenetre
        debut += taille - overlap
    return morceaux

# --- 2. Charger un corpus depuis un dossier de .md
def charger_corpus(dossier="./docs"):
    """Lit tous les .md d'un dossier et les chunke."""
    blocs = []
    chemin = Path(dossier)
    for fichier in chemin.rglob("*.md"):
        contenu = fichier.read_text(encoding="utf-8")
        for j, chunk in enumerate(chunk_text(contenu)):
            blocs.append({
                "id": f"{fichier.name}#chunk{j}",
                "source": fichier.name,
                "text": chunk
            })
    return blocs

# --- 3. Index vectoriel encapsule dans une classe
class IndexVectoriel:
    def __init__(self):
        self.modele = SentenceTransformer('all-MiniLM-L6-v2')
        self.blocs = []
        self.vecteurs = None

    def indexer(self, blocs):
        """Embeds tous les blocs et stocke."""
        self.blocs = blocs
        textes = [b["text"] for b in blocs]
        self.vecteurs = self.modele.encode(textes, normalize_embeddings=True)

    def chercher(self, question, k=3):
        """Renvoie les k blocs les plus pertinents avec leur score."""
        vec_q = self.modele.encode([question], normalize_embeddings=True)[0]
        sims = self.vecteurs @ vec_q  # produit scalaire = cosinus si normalises
        indices = sims.argsort()[::-1][:k]
        return [(self.blocs[i], float(sims[i])) for i in indices]

# --- 4. Prompt augmente
TEMPLATE = """Tu reponds en t'appuyant UNIQUEMENT sur le contexte ci-dessous.
Cite tes sources entre crochets [id_du_chunk].
Si la reponse n'est pas dans le contexte, dis : "Je ne sais pas".

CONTEXTE :
{contexte}

QUESTION : {question}
REPONSE :"""

def repondre(idx, question, k=3):
    """RAG complet : recherche + prompt augmente (LLM a brancher)."""
    hits = idx.chercher(question, k=k)
    contexte = "\n\n".join(f"[{b['id']}] {b['text']}" for b, _ in hits)
    prompt = TEMPLATE.format(contexte=contexte, question=question)
    print(prompt)  # a remplacer par appel a Ollama / OpenAI / Anthropic
    return prompt, hits

# --- Demo (a faire tourner si vous avez un dossier ./docs/)
# idx = IndexVectoriel()
# idx.indexer(charger_corpus("./docs"))
# repondre(idx, "Quelle est notre politique de teletravail ?", k=3)

### Évaluation suggérée (5 questions)

Pour chaque question, on évalue **manuellement** :

| Question | Bonne réponse ? | Source citée correcte ? | Pas d'invention ? |
|---|---|---|---|
| Q1 | ✅/❌ | ✅/❌ | ✅/❌ |
| Q2 | ✅/❌ | ✅/❌ | ✅/❌ |
| Q3 | ✅/❌ | ✅/❌ | ✅/❌ |
| Q4 | ✅/❌ | ✅/❌ | ✅/❌ |
| Q5 | ✅/❌ | ✅/❌ | ✅/❌ |

**Cible pédagogique** : ≥ 4/5 sur la première colonne. En dessous, c'est typiquement un problème de **chunking** (trop large/petit) ou un **k trop faible**.

---

## TP 3 Partie B — MCP marketplace : check-list de validation

### Configuration attendue (`claude_desktop_config.json`)

```json
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-filesystem", "/chemin/projet"]
    },
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": { "GITHUB_PERSONAL_ACCESS_TOKEN": "ghp_..." }
    }
  }
}
```

### Critères d'acceptation

- ✅ Le serveur `filesystem` apparaît dans l'IDE (icône outils).
- ✅ L'IA peut **lister les TODO** : appel `read_file` ou `search` sur `**/*.py`.
- ✅ L'IA peut **rédiger une PR** : `create_pull_request` réussit.
- ✅ Audit visible : trace des appels d'outils dans l'UI.

### Erreurs fréquentes

| Symptôme | Cause probable | Fix |
|---|---|---|
| « Tool not found » | npx pas dans le PATH | Installer Node ≥ 18, relancer l'IDE |
| Token refusé | PAT GitHub sans scopes `repo` | Régénérer un PAT avec `repo` + `read:org` |
| 0 TODO trouvé | Le pattern n'est pas dans le dossier exposé | Ajuster le chemin du `filesystem` |
| « MCP server failed » | Mauvais chemin absolu | Vérifier que le chemin existe |
